In [ ]:
# ==========================================
# 🧠 CAMPUS CONNECT: STABLE AI SERVER v2
# ==========================================

# 1. Install Dependencies
!pip install fastapi uvicorn pyngrok python-multipart transformers torch pillow nest_asyncio

import uvicorn
import getpass
import os
import io
import nest_asyncio
import torch
from fastapi import FastAPI, UploadFile, File
from pyngrok import ngrok
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# 2. Load the BLIP Model DIRECTLY (Fixes Pipeline Error)
print("⏳ Loading BLIP Model... (This takes ~30s)")

# Detect GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using Device: {device}")

try:
    # Manual loading ensures compatibility
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
    print("✅ BLIP Model Loaded & Ready!")
except Exception as e:
    print(f"❌ Error loading model: {e}")

# 3. Define the API
app = FastAPI()

@app.get("/")
def home():
    return {"message": "CampusConnect AI is Running!"}

@app.post("/analyze")
async def analyze(file: UploadFile = File(...)):
    try:
        # Read the uploaded image
        image_data = await file.read()
        image = Image.open(io.BytesIO(image_data)).convert("RGB")

        # Run Inference (Manual Method)
        inputs = processor(images=image, return_tensors="pt").to(device)
        
        # Generate caption
        out = model.generate(**inputs, max_new_tokens=50)
        caption = processor.decode(out[0], skip_special_tokens=True)

        # Simple logic to guess title/category
        title = caption.split(".")[0].title() 
        category = "Found" if "found" in caption.lower() else "Lost"

        print(f"🔍 Analyzed: {caption}")

        return {
            "title": title,
            "description": caption,
            "category": category
        }
    except Exception as e:
        return {"error": str(e)}

# 4. SECURE TOKEN INPUT
print("\n👇 Paste your Ngrok Authtoken below and hit Enter:")
NGROK_AUTH_TOKEN = getpass.getpass()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 5. Open the Tunnel
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("="*60)
print(f"🚀 YOUR PUBLIC URL IS:  {public_url}/analyze")
print("="*60)

# 6. Start the Server
nest_asyncio.apply()
config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)
await server.serve()